# Scene3D

## I. Getting Started Guide

### 1. Scene3D Model Introduction

Scene3D is a monocular depth estimation network within the VisionPilot suite designed to infer per-pixel scene depth from a single
front-view image. It follows an Encoder-Context-Decoder pattern, combining pretrained semantic features with a dedicated depth-context
module to recover geometry-aware structure.

The architecture components include:

- A pretrained Backbone (wrapped by `PreTrainedBackbone`) that extracts hierarchical multi-scale features from the input image. 
In this upstream stage, backbone parameters are frozen to preserve robust pretrained representations.

- A DepthContext module that consumes the deepest feature map, performs global spatial pooling, passes it through MLP layers, reshapes
the compact representation into a 2D context map, and expands it with convolutional layers.

- A residual attention-style fusion in `DepthContext` that injects context into deep features using $Context \times Features + Features$,
helping the model encode global scene layout while preserving local feature detail.

- A Scene3D Neck (decoder) with progressive transposed-convolution upsampling and skip connections from encoder stages (`features[3]`,
`features[2]`, `features[1]`) to recover spatial resolution and fine structural cues.

- A Scene3D Head that performs the final upsampling with an additional skip connection from `features[0]`, followed by convolutional
prediction layers that produce a single-channel dense depth output.

As a scene geometry model, Scene3D processes raw perspective frames and predicts dense depth for each pixel in real time. This allows
downstream driving stacks to estimate free space and object distance cues without requiring LiDAR, while remaining lightweight enough for
embedded deployment workflows.

![](../../Media/Scene3D_GIF_2.gif)

### 2. Environment Setup

**If you have done these steps of environment setup in other End-to-End Model Tutorials, please skip this and move forward to the next subsection `3. Download Models`.**

First, we need to clone the repository to your local environment. We will use the official repository from the
Autoware Foundation.

In [ ]:
import os

base_dir = os.getcwd()

# Clone Autoware VisionPilot repo, if not yet done
if not os.path.exists("./autoware_vision_pilot"):
    !git clone https://github.com/autowarefoundation/autoware_vision_pilot.git

The VisionPilot project relies on several key libraries, including **PyTorch** for model execution and
**OpenCV** for image processing.

We will use `pip` to install the required dependencies directly from the `requirements.txt` file provided in 
the `Models` directory.

In [ ]:
# Install required dependencies
%pip install --upgrade pip
%pip install -r autoware_vision_pilot/Models/requirements.txt

# Verify the installation (checking torch as a primary dependency)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### 3. Download Models

You can manually download the model weight files, using the links below. Subsequent tutorial sections assume
you already downloaded them and put them inside directory: 
`autoware_vision_pilot/Tutorials/E2E_Models/weights/`.

- [Link to Download Pytorch Model Weights (*.pth)](https://drive.google.com/file/d/1MrKhfEkR0fVJt-SdZEc0QwjwVDumPf7B/view?usp=sharing)
- [Link to Download Traced Pytorch Model (*.pt)](https://drive.google.com/file/d/1-LO3j2YCvwxeNLzyLrnzEwalTrYUZgK0/view?usp=drive_link)
- [Link to Download ONNX FP32 Weights (*.onnx)](https://drive.google.com/file/d/19gMPt_1z4eujo4jm5XKuH-8eafh-wJC6/view?usp=drive_link)

You can also use below script block, which uses `gdown` to download above model weights into such default directory.

In [ ]:
%pip install gdown

import gdown

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata = {
    "pth"   : {
        "url"       : "https://drive.google.com/uc?id=1MrKhfEkR0fVJt-SdZEc0QwjwVDumPf7B",
        "filepath"  : os.path.join(base_dir, model_dir, "scene3d.pth")
    },
    "pt"    : {
        "url"       : "https://drive.google.com/uc?id=1-LO3j2YCvwxeNLzyLrnzEwalTrYUZgK0",
        "filepath"  : os.path.join(base_dir, model_dir, "scene3d.pt")
    },
    "onnx"  : {
        "url"       : "https://drive.google.com/uc?id=19gMPt_1z4eujo4jm5XKuH-8eafh-wJC6",
        "filepath"  : os.path.join(base_dir, model_dir, "scene3d.onnx")
    }
}

for weight_type in metadata:

    url         = metadata[weight_type]["url"]
    filepath    = metadata[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading Scene3D {weight_type} weights...")
        gdown.download(url, filepath, quiet = False)
    else:
        print(f"Scene3D {weight_type} weights already exist at {filepath}. Skipping download.")

## II. Quick Test/Inference

Pre-requisites: please ensure that you have completed the above **Environment Setup**and **Download Models** first.

### 1. Image Inference

The `image_visualization.py` script facilitates batch processing of images by loading a pre-trained model checkpoint, performing a forward pass on each image, and overlaying colorized depth predictions onto the original frames.

#### a. Run Batch Image Inference

To run this in any environment, we must navigate to the specific visualization directory to ensure the script's internal import paths resolve correctly.

Key Script Parameters:
- `-p / --model_checkpoint_path`: location of your `.pth` file containing the trained weights.
- `-i / --input_image_dirpath`: directory containing `.png`, `.jpg`, or `.jpeg` files to process.
- `-o / --output_image_dirpath`: destination directory where the script saves visualization files as `[image_id]_result.png`.

In [ ]:
# 1. Path declaration
# Here we use the previously downloaded Pytorch weight file.
# For input and output directories, you can change them to your preferred paths.
# For now we use default sample image assets in the repository.
CHECKPOINT = metadata["pth"]["filepath"]
INPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/images/"
)
OUTPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/scene3d/images/"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/Scene3D && \
python3 image_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_DIR)} \
    -o {os.path.abspath(OUTPUT_DIR)}

#### b. Display Results in Notebook

After running the inference, we can use matplotlib and PIL to visualize the generated depth overlays directly within this notebook.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Path to the newly created results
result_files = sorted([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".png")
])

if (result_files):
    # Display the first 3 results
    fig, axes = plt.subplots(
        1, 3,
        figsize = (20, 10)
    )
    for i, ax in enumerate(axes):
        if (i < len(result_files)):
            img_path = os.path.join(OUTPUT_DIR, result_files[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Result: {result_files[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 2. Video Inference

In this subsection, we will apply the Scene3D model to video streams.
The `video_visualization.py` script processes video files frame-by-frame, performs depth inference, and compiles the results into a new annotated video file.

The video inference pipeline utilizes `OpenCV` for frame extraction/writing and uses a viridis colormap overlay for visualizing relative depth intensity.

Key script parameters:

- `-p / --model_checkpoint_path`: path to the trained Scene3D weights.
- `-i / --input_video_filepath`: path to the source video file.
- `-o / --output_video_filepath`: destination output path for the visualized video.
- `-v / --vis`: optional flag to enable a pop-up window showing frames as they are processed.

Technical details:

- The script resizes each frame to $(640, 320)$ before inference, then resizes visualization outputs to $(1280, 720)$ for the final video.
- It applies a viridis colormap and alpha blending with $\alpha = 0.97$ to produce a depth-overlay visualization.
- Video capture and writer pointers are released automatically upon completion or stream end.

In [ ]:
# 1. Path declaration
# Similarly as image inference above:
# - Pytorch weight file.
# - Input/Output paths can be changed to preferred.
# - Using defaults for now.
INPUT_VIDEO_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/videos/highway_normal.mp4"
)
OUTPUT_VIDEO_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/scene3d/videos/highway_normal_output"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/Scene3D && \
python3 video_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_VIDEO_PATH)} \
    -o {os.path.abspath(OUTPUT_VIDEO_PATH)}

## III. Model Training

### 1. Model Training on New Data

#### a. Load pre-trained model or load vanilla un-trained model for training from scratch

#### b. Prepare the training datasets

#### c. How to load dataset

#### d. How to run training

#### e. How to visualize training results

### 2. Building an Inference Pipeline

#### a. Using ROS2 to publish an image, run inference and visualize results

TBD.

#### b. Using IceOryx to publish an image, run inference and visualize results

TBD.

#### c. Using Zenoh to publish an image, run inference and visualize results

TBD.

#### d. Using Gstreamer to get an image, run inference and visualize results

TBD.